# Predict GPR carbon stocks for meadow polygons

Configure the input model, meadow polygons, output directory, and sediment depth below, then run the cells in order. This notebook sources and invokes `predict_gpr_at_maps.R`; it does not reimplement or define the prediction functions. Run it from the v2 R environment with the required NetCDF rasters available under `data/env_rasters/`.

In [ ]:
# User configuration; paths are relative to codebase/v2/src.
project_path <- Sys.getenv("SEAGRASS_V2_ROOT", "")
model_path <- file.path("data", "review", "GPR_final.rds")
meadow_path <- file.path("data", "seagrass_eov_poly_2025.csv")
output_dir <- file.path("output", "pixel_grouped", "stocks")
depth_cm <- 100

# Locate the v2 source root from the configured path or notebook working directory.
kernel_cwd <- getwd()
start_dirs <- if (nzchar(project_path)) path.expand(project_path) else kernel_cwd
offsets <- c("", "src", file.path("codebase", "v2", "src"))
checked <- character()
project_root <- NA_character_
for (start in start_dirs) {
  directory <- normalizePath(start, winslash = "/", mustWork = FALSE)
  for (i in seq_len(40L)) {
    for (offset in offsets) {
      candidate <- file.path(directory, offset)
      checked <- c(checked, candidate)
      markers <- file.path(candidate, c(
        "predict_gpr_at_maps.R",
        file.path("modelling", "R", "init_repo.R"),
        file.path("modelling", "R", "project_root.R")
      ))
      if (all(file.exists(markers))) {
        project_root <- normalizePath(candidate, winslash = "/", mustWork = TRUE)
        break
      }
    }
    if (!is.na(project_root)) break
    parent <- dirname(directory)
    if (identical(parent, directory)) break
    directory <- parent
  }
  if (!is.na(project_root)) break
}
if (is.na(project_root)) {
  stop(
    "Cannot find the v2 project root. Set SEAGRASS_V2_ROOT to the clone root, codebase/v2, or codebase/v2/src.",
    "\nKernel working directory: ", kernel_cwd,
    "\nChecked:\n  ", paste(unique(checked), collapse = "\n  "),
    call. = FALSE
  )
}
setwd(project_root)
cat("v2 project root:", project_root, "\n")

In [ ]:
# Source the original implementation and invoke its map-prediction entry point.
script_path <- file.path(project_root, "predict_gpr_at_maps.R")
source(script_path, local = .GlobalEnv)
prediction_args <- c(model_path, meadow_path, output_dir, as.character(depth_cm))
predicted_meadows <- predict_gpr_at_maps(prediction_args)

In [ ]:
# Review prediction coverage and generated files.
if (!exists("predicted_meadows")) {
  stop("Run the prediction cell first.", call. = FALSE)
}
print(table(predicted_meadows$prediction_status))
cat("Map and prediction tables saved under:", file.path(project_root, output_dir), "\n")